In [1]:
import numpy as np
import scipy.stats as stats
import src.noise as noise
from src.core import NoisyFloat
from src.analysis import NoisyContingencyTable

Here's a contingency table.

In [2]:
true_tbl = np.array([
    [65, 109],
    [243, 1348]
])

It's straightforward to have `scipy` calculate the 95% confidence interval of its odds ratio.

In [3]:
scipy_ci = stats.contingency.odds_ratio(true_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.363633, 4.629785)


The `dpvalue` library can do this too. Although it uses a _credible interval_ because it's drawing from a posterior. It implicitly converts `true_tbl` into a table with elements of type `NoisyFloat` (the noise is just zero).

In [4]:
nct = NoisyContingencyTable(true_tbl)
dpvalue_lo, dpvalue_hi = nct.with_sampling_uncertainty().odds_ratio().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.971353, 3.731481)


More or less consistent, I guess.

Let's add some noise and pretend the contingency table is a differentially private release.

In [5]:
noise_factory = noise.gaussian(loc=0.0, scale=12.0)
noisy_tbl = np.vectorize(lambda x: NoisyFloat.draw(x, noise_factory()))(true_tbl)
noisy_tbl

array([[~56.46419055982567, ~97.25752060741455],
       [~233.2367124277515, ~1338.3747665517387]], dtype=object)

What would `scipy` do with this? Since `scipy` isn't DP-noise-aware, it can only take the observed values as truth and account for sampling uncertainty as it did before.

But oops, we can't use `scipy` on this directly, even though `NoisyFloat` is castable to `float`. The elements are counts, and have to be type `int`. Let's extract the noisy observations.

In [6]:
noisy_int_tbl = [
    [int(noisy_tbl[0, 0]._obs), int(noisy_tbl[0, 1]._obs)],
    [int(noisy_tbl[1, 0]._obs), int(noisy_tbl[1, 1]._obs)]]

noisy_int_tbl

[[56, 97], [233, 1338]]

And here's the `scipy` confidence interval again.

In [7]:
scipy_ci = stats.contingency.odds_ratio(noisy_int_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.319567, 4.738337)


But the `dpvalue` library can use `noisy_tbl` directly. It accounts for both uncertainty from DP noise _and_ uncertainty from sampling.

In [8]:
nct_noisy = NoisyContingencyTable(noisy_tbl)
dpvalue_lo, dpvalue_hi = nct_noisy.with_sampling_uncertainty().odds_ratio().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.578220, 4.519874)


But it takes almost 8 seconds to compute this...

We can get a chi squared statistic the same way. Note the three "levels" of this. First the statistic itself without any noise.

In [9]:
noisy_table_no_posteriors = NoisyContingencyTable([
    [noisy_tbl[0, 0]._obs, noisy_tbl[0, 1]._obs],
    [noisy_tbl[1, 0]._obs, noisy_tbl[1, 1]._obs]])

dpvalue_lo, dpvalue_hi = noisy_table_no_posteriors.chi_squared().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (48.026907, 48.026907)


Differential privacy noise only:

In [10]:
nct = NoisyContingencyTable(noisy_tbl)
dpvalue_lo, dpvalue_hi = nct.chi_squared().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (9.235601, 109.015835)


Sampling noise only:

In [11]:
dpvalue_lo, dpvalue_hi = noisy_table_no_posteriors.with_sampling_uncertainty().chi_squared().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (38.823024, 56.218972)


Differential privacy noise and sampling noise. This is what we raelly want.

In [12]:
dpvalue_lo, dpvalue_hi = nct.with_sampling_uncertainty().chi_squared().credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (31.226197, 70.256242)
